# SegOid — Post-Segmentation Analysis

Group, filter, and export a SegOid metrics CSV into a multi-tab Excel ready for inspection or GraphPad import.

**Prerequisite:** the metrics CSV must come from a SegOid run with **"Parse filename into fields"** enabled in the GUI. That step turns image filenames into metadata columns (e.g., `condition`, `parameters`) that this notebook groups by. If you skipped that step, re-run inference with the option enabled and re-export.

**Privacy note:** uploading the metrics CSV here uses Google's servers, the same as any other cloud service. The CSV contains numerical metrics and filenames — no images. Check with your institution if your data has any restrictions on using cloud platforms before uploading.

## 1. Install SegOid

In [ ]:
!pip install --quiet git+https://github.com/Rtasseff/SegOid.git@main

## 2. Upload your metrics CSV

Run the cell. A file picker appears — choose your `metrics.csv` (or whatever SegOid named it).

In [ ]:
from google.colab import files
uploaded = files.upload()
metrics_path = next(iter(uploaded))
print(f'Uploaded: {metrics_path}')

## 3. Configure the analysis

Edit the values below for this run. Set any filter to `None` to disable it.

- **`group_by`**: list of column names already in the metrics CSV. The output Excel gets one tab per group.
- **Filter values**: numeric thresholds, or `None` to skip the filter.

In [ ]:
# === EDIT THESE ===
group_by = ['condition']         # e.g., ['condition'] or ['condition', 'parameters']

# Filters: None = filter off (everything passes). Set a number to enable.
# Default is "all off" so you see all your data first; then add filters once you
# know what range of values your spheroids actually have for THIS dataset.
# Tip: pixel-area thresholds are very sensitive to magnification — a value that
# works at one zoom level will exclude everything at another. Prefer the µm²
# thresholds when you've set a pixel size in SegOid.
circularity_min = None           # e.g. 0.5 to drop irregular shapes
area_min_px = None               # e.g. 10000 to drop tiny detections
area_max_px = None               # e.g. 50000 to drop huge detections
area_min_um2 = None              # use µm² instead of px when SegOid had a pixel size
area_max_um2 = None
mad_max = None                   # robust outlier filter on area (in MAD units); e.g. 3 to drop outliers

output_filename = 'post_seg.xlsx'
# ===================

## 4. Run the analysis

In [ ]:
from pathlib import Path
from src.post_segmentation import PostSegConfig, run

config = PostSegConfig(
    metrics_csv=Path(metrics_path),
    output_xlsx=Path(output_filename),
    group_by=group_by,
    circularity_min=circularity_min,
    area_min_px=area_min_px,
    area_max_px=area_max_px,
    area_min_um2=area_min_um2,
    area_max_um2=area_max_um2,
    mad_max=mad_max,
)
out = run(config)
print(f'Wrote {out}')

## 5. Download the Excel

In [ ]:
files.download(output_filename)